In [3]:
# pip install
#!pip install pandas numpy argon2-cffi


%pip install pandas numpy argon2-cffi

# Used Argon2-cffi to prevent build issues



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# authentication method

# Import Required Packages
import json
import time
import os
import secrets
import hmac
import hashlib
from argon2 import PasswordHasher
from argon2.exceptions import VerifyMismatchError

# argon2 hasher
ph = PasswordHasher()
#constants
DB_FILE = 'users_database.json'
MAX_ATTEMPTS = 5
LOCKOUT_DURATION = 900
OTP_EXPIRY = 60


#loading user database
def load_user_data():
    if not os.path.exists(DB_FILE):
        print("No user database found.")
        return {} #empty dictionary
    with open(DB_FILE, 'r') as file:
        return json.load(file)


# take user database and dump in JSON file
def save_user_data(data):
    with open(DB_FILE, 'w') as file:
        json.dump(data, file, indent=4)


# save user data in database
def register_user(username, password):
    users = load_user_data()

    #check if user is already registered
    if username in users:
        print("Error: User already exists.")
        return False

    #check if password meets requirements
    if len(password) < 16 or password.lower() == username.lower():
        print("Error: password must be 16+ chars and not your username.")
        return False

    #hash the password
    pw_hash = ph.hash(password)

    #Data chunk for new user to be saved to db
    users[username] = {
        "password": pw_hash,
        "failed_attempts": 0,
        "lockout_until": 0,
        "totp_secret": secrets.token_hex(16),
        "current_otp": None,
        "otp_expiry": 0
    }

    save_user_data(users)
    print(f"User {username} registered successfully.")
    return True


#Verifies user login info and handles lockout scenario
def login_user(username, password):
    users = load_user_data()

    #check if user is in the db
    if username not in users:
        print("User not found.")
        return False

    #find user entry in db and get timestamp
    user = users[username]
    now = time.time()

    #check lockout time (if there is one or if expired)
    if now < user['lockout_until']:
        remaining = int((user['lockout_until'] - now) // 60)
        print(f"Account locked. Try again in {remaining} minutes.")
        return False

    # argon2 constant time comparison
    try:
        ph.verify(user['password'], password)

        # Reset attempts on success
        user['failed_attempts'] = 0
        save_user_data(users)
        return True
    #if failed attempt increment failure and set lockout
    except VerifyMismatchError:
        user['failed_attempts'] += 1
        print(f"Invalid password. Attempt {user['failed_attempts']}/{MAX_ATTEMPTS}")

        if user['failed_attempts'] >= MAX_ATTEMPTS: #five lockout attempts
            user['lockout_until'] = now + LOCKOUT_DURATION #15 minute lockout time
            print("Account locked for 15 minutes.")

        #save to file
        save_user_data(users)
        return False
    #memory cleanup of plaintext pw (just in case)
    finally:
        del password


#generate an OTP for login authentication
def generate_totp(username):
    #grab user information from db
    users = load_user_data()
    user = users[username]

    #create 6 digit otp using HMAC
    msg = str(int(time.time())).encode()
    key = user['totp_secret'].encode()
    h = hmac.new(key, msg, hashlib.sha256).hexdigest()
    otp = str(int(h, 16))[-6:]

    #store in user's entry in db
    user['current_otp'] = otp
    user['otp_expiry'] = time.time() + OTP_EXPIRY
    save_user_data(users)

    #simulate auth app in terminal
    print(f"\n[AUTHENTICATOR APP] Your 2FA Code: {otp} (Expires in 60s)")
    return otp


#verify the user's otp
def verify_totp(username, input_otp):
    #grab user information from db
    users = load_user_data()
    user = users[username]

    #check db entry for otp and expiry time
    if user['current_otp'] is None or time.time() > user['otp_expiry']:
        print("OTP has expired or does not exist.")
        return False

    #constant-time comparison for freshness
    if hmac.compare_digest(user['current_otp'], input_otp):
        #erase/override current otp so it cannot be used again
        user['current_otp'] = None
        save_user_data(users)
        return True
    else:
        print("Invalid OTP.")
        return False

In [5]:
# main 

# heyyy vinnie sorry i went ahead and finished the whole of authentication
def main ():
    print()

if __name__ == "__main__":
    main()